<a href="https://colab.research.google.com/github/abhirupchak/Wheat-Yield-Regression-Analysis-FAOSTAT/blob/main/knn_wine_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# 1. Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Load dataset
df = pd.read_csv(file_name)

# 3. Separate features and target
X = df.drop(columns=["class"])
y = df["class"]

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Build KNN pipeline
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]
)

# 6. Train model
model.fit(X_train, y_train)

# 7. Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# 8. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("DATASET SHAPE:", df.shape)

print("\nCONFUSION MATRIX")
cm_df = pd.DataFrame(
    cm,
    index=["Actual Class 1", "Actual Class 2", "Actual Class 3"],
    columns=["Predicted Class 1", "Predicted Class 2", "Predicted Class 3"]
)
display(cm_df)

# 9. Calculate one-vs-rest TP, TN, FP, FN
TP = []
TN = []
FP = []
FN = []

for i in range(len(cm)):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - (tp + fn + fp)

    TP.append(tp)
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)

# 10. Overall metrics
accuracy = accuracy_score(y_test, y_pred)
error = 1 - accuracy

precision = precision_score(
    y_test, y_pred, average="weighted"
)

recall = recall_score(
    y_test, y_pred, average="weighted"
)

specificity = sum(
    TN[i] / (TN[i] + FP[i])
    for i in range(len(TN))
) / len(TN)

f1 = f1_score(
    y_test, y_pred, average="weighted"
)

auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

# 11. Display evaluation metrics
results = pd.DataFrame({
    "Measure": [
        "Accuracy",
        "TP",
        "TN",
        "FP",
        "FN",
        "Error",
        "Recall",
        "Specificity",
        "F1 Score",
        "AUC"
    ],
    "Score": [
        accuracy,
        sum(TP),
        sum(TN),
        sum(FP),
        sum(FN),
        error,
        recall,
        specificity,
        f1,
        auc
    ]
})

print("\nEVALUATION MEASURES")
display(results)

# 12. Classification Report
print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "Class 1",
        "Class 2",
        "Class 3"
    ]
))

Saving Wine dataset.csv to Wine dataset.csv
DATASET SHAPE: (178, 14)

CONFUSION MATRIX


,Predicted Class 1,Predicted Class 2,Predicted Class 3
Actual Class 1,12,0,0
Actual Class 2,0,13,1
Actual Class 3,0,0,10



EVALUATION MEASURES


,Measure,Score
0,Accuracy,0.972222
1,TP,35.000000
2,TN,71.000000
3,FP,1.000000
4,FN,1.000000
5,Error,0.027778
6,Recall,0.972222
7,Specificity,0.987179
8,F1 Score,0.972369
9,AUC,0.998834



CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Class 1       1.00      1.00      1.00        12
     Class 2       1.00      0.93      0.96        14
     Class 3       0.91      1.00      0.95        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36

